# use smplest_py310 env

In [ ]:
if "_magic_done" not in globals(): # prevent multiple run
    %load_ext autoreload
    %autoreload 2
    %cd ..
    _magic_done = True

In [ ]:
from pathlib import Path

In [ ]:
import os
import os.path as osp
import argparse
import numpy as np
import torchvision.transforms as transforms
import torch.backends.cudnn as cudnn
import torch
import cv2
import datetime
from tqdm import tqdm
from pathlib import Path
from human_models.human_models import SMPLX
from ultralytics import YOLO
from main.base import Tester
from main.config import Config
from utils.data_utils import load_img, process_bbox, generate_patch_image
from utils.visualization_utils import render_mesh
from utils.inference_utils import non_max_suppression

In [ ]:
fps = 30
args_ckpt_name = "smplest_x_h"
args_start = 1
# args_end =  # number of images
args_multi_person = False

In [ ]:

# init config
# root_dir = Path.cwd().resolve().parent.parent
root_dir = "."
config_path = osp.join('./pretrained_models', args_ckpt_name, 'config_base.py')
cfg = Config.load_config(config_path)

checkpoint_path = osp.join('./pretrained_models', args_ckpt_name, f'{args_ckpt_name}.pth.tar')
time_str = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
exp_name = f'inference_{args_ckpt_name}_{time_str}'

new_config = {
    "model": {
        "pretrained_model_path": checkpoint_path,
    },
    "log":{
        'exp_name':  exp_name,
        'log_dir': osp.join(root_dir, 'outputs', exp_name, 'log'),  
        }
}
cfg.update_config(new_config)
cfg.prepare_log()

# init human models
smpl_x = SMPLX(cfg.model.human_model_path)

# init tester
demoer = Tester(cfg)
demoer.logger.info(f"Using 1 GPU.")
demoer._make_model()

# init detector
bbox_model = getattr(cfg.inference.detection, "model_path", 
                    './pretrained_models/yolov8x.pt')
detector = YOLO(bbox_model)

In [ ]:
import os
import sys
import subprocess
import shlex
import shutil



file_name = "Abe_proud_03_000.mp4"


name, ext = os.path.splitext(file_name)
args_file_name = name
ext = ext.lstrip('.')  # Remove the leading dot

img_path = os.path.join('./demo/input_frames', name)
output_path = os.path.join('./demo/output_frames', name)

os.makedirs(img_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

# convert video to frames
if ext in ['mp4', 'avi', 'mov', 'mkv', 'flv', 'wmv', 'webm', 'mpeg', 'mpg']:
    ffmpeg_cmd = shlex.split(f"ffmpeg -i ./demo/{file_name} -f image2 -vf fps={fps}/1 -qscale 0 {img_path}/%06d.jpg")
    subprocess.run(ffmpeg_cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
elif ext in ['jpg', 'jpeg', 'png', 'bmp', 'gif', 'tiff', 'tif', 'webp', 'svg']:
    src = os.path.join('./demo', file_name)
    dst = os.path.join(img_path, f"000001.{ext}")
    shutil.copy(src, dst)
else:
    print("Unknown file type.")
    sys.exit(1)

end_count = len([f for f in os.listdir(img_path) if os.path.isfile(os.path.join(img_path, f))])
print(end_count)
args_end = end_count

In [ ]:

img_folder = osp.join(root_dir, 'demo', 'input_frames', args_file_name)
output_folder = osp.join(root_dir, 'demo', 'output_frames', args_file_name)
os.makedirs(output_folder, exist_ok=True)
start = int(args_start)
end = int(args_end) + 1



for frame in tqdm(range(start, end)):
    #####################################################################################################
    # Collect SMPL outputs for each frame instead of saving them individually
    if frame == start:
        motion_seq = {
            "smplx_root_pose": [],
            "smplx_body_pose": [],
            "smplx_lhand_pose": [],
            "smplx_rhand_pose": [],
            "smplx_jaw_pose": [],
            "smplx_shape": [],

            "cam_trans" : [],
            "smplx_joint_proj" : [],
            "smplx_mesh_cam" : [],
            "smplx_expr" : [],
            "smplx_joint_cam" : [],


            "ignored_frames": [],
        }

    #####################################################################################################
    
    # prepare input image
    img_path =osp.join(img_folder, f'{int(frame):06d}.jpg')

    transform = transforms.ToTensor()
    original_img = load_img(img_path)
    vis_img = original_img.copy()
    original_img_height, original_img_width = original_img.shape[:2]
    
    # detection, xyxy
    yolo_bbox = detector.predict(original_img, 
                            device='cuda', 
                            classes=00, 
                            conf=cfg.inference.detection.conf, 
                            save=cfg.inference.detection.save, 
                            verbose=cfg.inference.detection.verbose
                                )[0].boxes.xyxy.detach().cpu().numpy()

    if len(yolo_bbox)<1:
        # save original image if no bbox
        num_bbox = 0
        motion_seq["ignored_frames"].append(frame)
    # if not args_multi_person: 
    elif not args_multi_person:
        # only select the largest bbox
        num_bbox = 1
        # yolo_bbox = yolo_bbox[0]
    else:
        # keep bbox by NMS with iou_thr
        yolo_bbox = non_max_suppression(yolo_bbox, cfg.inference.detection.iou_thr)
        num_bbox = len(yolo_bbox)

    # loop all detected bboxes
    for bbox_id in range(num_bbox):
        yolo_bbox_xywh = np.zeros((4))
        yolo_bbox_xywh[0] = yolo_bbox[bbox_id][0]
        yolo_bbox_xywh[1] = yolo_bbox[bbox_id][1]
        yolo_bbox_xywh[2] = abs(yolo_bbox[bbox_id][2] - yolo_bbox[bbox_id][0])
        yolo_bbox_xywh[3] = abs(yolo_bbox[bbox_id][3] - yolo_bbox[bbox_id][1])
        
        # xywh
        bbox = process_bbox(bbox=yolo_bbox_xywh, 
                            img_width=original_img_width, 
                            img_height=original_img_height, 
                            input_img_shape=cfg.model.input_img_shape, 
                            ratio=getattr(cfg.data, "bbox_ratio", 1.25))                
        img, _, _ = generate_patch_image(cvimg=original_img, 
                                            bbox=bbox, 
                                            scale=1.0, 
                                            rot=0.0, 
                                            do_flip=False, 
                                            out_shape=cfg.model.input_img_shape)
            
        img = transform(img.astype(np.float32))/255
        img = img.cuda()[None,:,:,:]
        inputs = {'img': img}
        targets = {}
        meta_info = {}

        # mesh recovery
        with torch.no_grad():
            out = demoer.model(inputs, targets, meta_info, 'test')

        mesh = out['smplx_mesh_cam'].detach().cpu().numpy()[0]

        #####################################################################################################
        for key in [
            # 'img',
            'cam_trans',
            'smplx_joint_proj',

            'smplx_mesh_cam',
            'smplx_root_pose',
            'smplx_body_pose',
            'smplx_lhand_pose',
            'smplx_rhand_pose',
            'smplx_jaw_pose',

            'smplx_shape',
            'smplx_expr',
            'smplx_joint_cam'
            ] :

            motion_seq[key].append(out[key].detach().cpu().numpy()[0])
        #####################################################################################################




##################################################################################################################
# === After finishing the per-image loop ===
print("Merging SMPL parameters into single motion sequence file...")

# Convert lists to arrays [T, ...]
for k in motion_seq:
    motion_seq[k] = np.stack(motion_seq[k], axis=0)

# Save single motion clip
seq_path = osp.join(output_folder, "motion_sequence.npz")
np.savez(seq_path, **motion_seq)

print(f"✅ Saved merged motion sequence -> {seq_path}")
##################################################################################################################


In [ ]:
shutil.rmtree(img_folder)

In [ ]:
a = np.load("./demo/output_frames/Abe_proud_03_000/motion_sequence.npz")

In [ ]:
print(a['smplx_mesh_cam'].shape)
print(a['smplx_body_pose'].shape)
print(a['smplx_lhand_pose'].shape)
print(a['smplx_jaw_pose'].shape)
print(a['smplx_joint_cam'].shape)
print(a['smplx_root_pose'].shape)
print(a['smplx_rhand_pose'].shape)
print(a['smplx_joint_proj'].shape)
print(a['smplx_expr'].shape)
print(a['cam_trans'].shape)
print(a['smplx_shape'].shape)
print("ignorede frames:", a["ignored_frames"])

In [ ]:
def f(file_name, ):
    name, ext = os.path.splitext(file_name)
    args_file_name = name
    ext = ext.lstrip('.')  # Remove the leading dot

    img_path = os.path.join('./demo/input_frames', name)
    output_path = os.path.join('./demo/output_frames_v0', name)

    os.makedirs(img_path, exist_ok=True)
    os.makedirs(output_path, exist_ok=True)

    # convert video to frames
    if ext in ['mp4', 'avi', 'mov', 'mkv', 'flv', 'wmv', 'webm', 'mpeg', 'mpg']:
        ffmpeg_cmd = shlex.split(f"ffmpeg -i ./demo/Dataset_v0/{file_name} -f image2 -vf fps={fps}/1 -qscale 0 {img_path}/%06d.jpg")
        subprocess.run(ffmpeg_cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    elif ext in ['jpg', 'jpeg', 'png', 'bmp', 'gif', 'tiff', 'tif', 'webp', 'svg']:
        src = os.path.join('./demo/Dataset_v0', file_name)
        dst = os.path.join(img_path, f"000001.{ext}")
        shutil.copy(src, dst)
    else:
        print("Unknown file type.")
        sys.exit(1)

    end_count = len([f for f in os.listdir(img_path) if os.path.isfile(os.path.join(img_path, f))])
    args_end = end_count

    img_folder = osp.join(root_dir, 'demo', 'input_frames', args_file_name)
    output_folder = osp.join(root_dir, 'demo', 'output_frames_v0', args_file_name)
    os.makedirs(output_folder, exist_ok=True)
    start = int(args_start)
    end = int(args_end) + 1



    for frame in tqdm(range(start, end)):
        #####################################################################################################
        # Collect SMPL outputs for each frame instead of saving them individually
        if frame == start:
            motion_seq = {
                "smplx_root_pose": [],
                "smplx_body_pose": [],
                "smplx_lhand_pose": [],
                "smplx_rhand_pose": [],
                "smplx_jaw_pose": [],
                "smplx_shape": [],

                "cam_trans" : [],
                "smplx_joint_proj" : [],
                "smplx_mesh_cam" : [],
                "smplx_expr" : [],
                "smplx_joint_cam" : [],


                "ignored_frames": [],
            }

        #####################################################################################################
        
        # prepare input image
        img_path =osp.join(img_folder, f'{int(frame):06d}.jpg')

        transform = transforms.ToTensor()
        original_img = load_img(img_path)
        vis_img = original_img.copy()
        original_img_height, original_img_width = original_img.shape[:2]
        
        # detection, xyxy
        yolo_bbox = detector.predict(original_img, 
                                device='cuda', 
                                classes=00, 
                                conf=cfg.inference.detection.conf, 
                                save=cfg.inference.detection.save, 
                                verbose=cfg.inference.detection.verbose
                                    )[0].boxes.xyxy.detach().cpu().numpy()

        if len(yolo_bbox)<1:
            # save original image if no bbox
            num_bbox = 0
            motion_seq["ignored_frames"].append(frame)
        # if not args_multi_person: 
        elif not args_multi_person:
            # only select the largest bbox
            num_bbox = 1
            # yolo_bbox = yolo_bbox[0]
        else:
            # keep bbox by NMS with iou_thr
            yolo_bbox = non_max_suppression(yolo_bbox, cfg.inference.detection.iou_thr)
            num_bbox = len(yolo_bbox)

        # loop all detected bboxes
        for bbox_id in range(num_bbox):
            yolo_bbox_xywh = np.zeros((4))
            yolo_bbox_xywh[0] = yolo_bbox[bbox_id][0]
            yolo_bbox_xywh[1] = yolo_bbox[bbox_id][1]
            yolo_bbox_xywh[2] = abs(yolo_bbox[bbox_id][2] - yolo_bbox[bbox_id][0])
            yolo_bbox_xywh[3] = abs(yolo_bbox[bbox_id][3] - yolo_bbox[bbox_id][1])
            
            # xywh
            bbox = process_bbox(bbox=yolo_bbox_xywh, 
                                img_width=original_img_width, 
                                img_height=original_img_height, 
                                input_img_shape=cfg.model.input_img_shape, 
                                ratio=getattr(cfg.data, "bbox_ratio", 1.25))                
            img, _, _ = generate_patch_image(cvimg=original_img, 
                                                bbox=bbox, 
                                                scale=1.0, 
                                                rot=0.0, 
                                                do_flip=False, 
                                                out_shape=cfg.model.input_img_shape)
                
            img = transform(img.astype(np.float32))/255
            img = img.cuda()[None,:,:,:]
            inputs = {'img': img}
            targets = {}
            meta_info = {}

            # mesh recovery
            with torch.no_grad():
                out = demoer.model(inputs, targets, meta_info, 'test')

            mesh = out['smplx_mesh_cam'].detach().cpu().numpy()[0]

            #####################################################################################################
            for key in [
                # 'img',
                'cam_trans',
                'smplx_joint_proj',

                'smplx_mesh_cam',
                'smplx_root_pose',
                'smplx_body_pose',
                'smplx_lhand_pose',
                'smplx_rhand_pose',
                'smplx_jaw_pose',

                'smplx_shape',
                'smplx_expr',
                'smplx_joint_cam'
                ] :

                motion_seq[key].append(out[key].detach().cpu().numpy()[0])
            #####################################################################################################




    ##################################################################################################################
    # === After finishing the per-image loop ===
    print("Merging SMPL parameters into single motion sequence file...")

    # Convert lists to arrays [T, ...]
    for k in motion_seq:
        if len (motion_seq[k]) > 0:
            motion_seq[k] = np.stack(motion_seq[k], axis=0)
        else:
            motion_seq[k] = np.array([])

    # Save single motion clip
    seq_path = osp.join(output_folder, "motion_sequence.npz")
    np.savez(seq_path, **motion_seq)

    print(f"✅ Saved merged motion sequence -> {seq_path}")
    ##################################################################################################################


    shutil.rmtree(img_folder)

file_name = "Abe_proud_03_000.mp4"
f(file_name)

In [ ]:
file_names = os.listdir("./demo/Dataset_v0")
for idx, file_name in enumerate(file_names):
    name, ext = os.path.splitext(file_name)
    if os.path.exists("./demo/output_frames_v0/" + name + "/motion_sequence.npz"):
        continue

    print("[%d/%d]" % (
        idx + 1,
        len(file_names)
    ))

    f(file_name)
    print("-" * 100)